# NumCompute Demo – End‑to‑End Pipeline

This notebook demonstrates:
- Loading a CSV file with `io.load_csv` (including missing values)
- Preprocessing: scaling, one‑hot encoding, imputation
- Statistics: mean, std/variance, histogram, quantiles
- Sorting / searching (top‑k, binary search)
- Ranking and percentiles
- Finite‑difference gradient approximation
- Performance benchmark (vectorised vs. Python loop)

In [ ]:
# Incase, numpy import fails please run the command below
import sys
!{sys.executable} -m pip install numpy

In [11]:
import sys
sys.path.append('..')

import numpy as np
from numcompute.io import load_csv
from numcompute.preprocessing import StandardScaler, OneHotEncoder, SimpleImputer
from numcompute.stats import mean,std,  histogram, quantile
from numcompute.sort_search import topk, binary_search, quickselect
from numcompute.rank import rank, percentile
from numcompute.optim import grad
from numcompute.pipeline import FeatureUnion, Pipeline
from numcompute.metrics import accuracy, precision, recall, f1, confusion_matrix, mse, roc_curve, auc


In [3]:
# %% [markdown]
# # 1. Load the CSV file

# %%
csv_path = "dataset/test_student_performance_data.csv" 
data, column_names = load_csv(csv_path, skip_header=True)
print(f"Shape of loaded data: {data.shape}")
print(f"Column names: {column_names}")
  


# Check for missing values
missing_counts = np.isnan(data).sum(axis=0)
print("Missing values per column:", missing_counts)

Shape of loaded data: (2392, 15)
Column names: ['StudentID', 'Age', 'Gender', 'Ethnicity', 'ParentalEducation', 'StudyTimeWeekly', 'Absences', 'Tutoring', 'ParentalSupport', 'Extracurricular', 'Sports', 'Music', 'Volunteering', 'GPA', 'GradeClass']
Missing values per column: [0 1 0 2 0 2 5 1 0 1 1 0 0 2 1]


In [4]:
# %% [markdown]
# # 2. Preprocessing Define numeric and categorical features (based on the dataset)

# %%
numeric_features = ['Age', 'StudyTimeWeekly', 'Absences', 'GPA']
categorical_features = [
    'Gender', 'Ethnicity', 'ParentalEducation', 'Tutoring',
    'ParentalSupport', 'Extracurricular', 'Sports', 'Music', 'Volunteering'
]

# Get column indices
num_idx = [column_names.index(col) for col in numeric_features]
cat_idx = [column_names.index(col) for col in categorical_features]

X_num = data[:, num_idx]
X_cat = data[:, cat_idx]

# Impute numeric missing values (if any) with mean
imputer_num = SimpleImputer(strategy='mean')
X_num_imputed = imputer_num.fit_transform(X_num)

# Scale numeric features
scaler = StandardScaler()
X_num_scaled = scaler.fit_transform(X_num_imputed)

# Impute categorical missing values with a constant (e.g., -1)
imputer_cat = SimpleImputer(strategy='constant', fill_value=-1)
X_cat_imputed = imputer_cat.fit_transform(X_cat)

# One‑hot encode categorical features
encoder = OneHotEncoder(handle_unknown='ignore')
X_cat_encoded = encoder.fit_transform(X_cat_imputed)

# Combine
X_processed = np.hstack([X_num_scaled, X_cat_encoded])
print(f"Preprocessed data shape: {X_processed.shape}")

Preprocessed data shape: (2392, 34)


In [5]:
# %% [markdown]
# # 3. Statistics on GPA

# %%
gpa = X_num_imputed[:, 3]   # GPA column (last numeric)
print(f"Mean GPA: {mean(gpa):.4f}")
print(f"Std GPA : {std(gpa):.4f}")
print(f"25th percentile: {quantile(gpa, 0.25):.4f}")
print(f"Median (50th)   : {quantile(gpa, 0.5):.4f}")
print(f"75th percentile: {quantile(gpa, 0.75):.4f}")

counts, edges = histogram(gpa, n_bins=10)
print("Histogram counts (10 bins):", counts)

Mean GPA: 1.9054
Std GPA : 0.9139
25th percentile: 1.1748
Median (50th)   : 1.8942
75th percentile: 2.6218
Histogram counts (10 bins): [112 198 309 332 324 333 324 251 171  38]


In [6]:
# %% [markdown]
# # 4. Sorting and searching using real modules

# %%
print("Top 5 GPAs (values):", topk(gpa, 5, return_indices=False, sorted=True))

sorted_gpa = np.sort(gpa)
median_val = quantile(gpa, 0.5)
k = 499
kth_value = quickselect(gpa,k)
print(f"The {k+1}th smallest GPA is: {kth_value: .4f}")
idx, exists = binary_search(sorted_gpa, median_val)
print(f"Median GPA {median_val:.4f} found at sorted index: {idx} (exists: {exists})")

Top 5 GPAs (values): [4. 4. 4. 4. 4.]
The 500th smallest GPA is:  1.0410
Median GPA 1.8942 found at sorted index: 1196 (exists: False)


In [7]:
# %% [markdown]
# # 5. Ranking and percentiles using real modules

# %%
sample = gpa[:10]
print("Sample GPA:", sample)
print("Ranks (average method):", rank(sample, method="average"))
print("90th percentile (linear):", percentile(gpa, 90, interpolation="linear"))


Sample GPA: [2.92919559 3.04291483 0.11260225 2.05421814 1.28806118 3.08418361
 2.74823741 1.36014271 2.89681919 3.57347421]
Ranks (average method): [ 7.  8.  1.  4.  2.  9.  5.  3.  6. 10.]
90th percentile (linear): 3.1311345757833364


In [8]:
# %% [markdown]
# # 6. Gradient estimation using real grad function

# %%
def func(x):
    return x[0]**2 + np.sin(x[1])

point = np.array([2.0, 0.5])
grad_val = grad(func, point, h=1e-6, method='central')
print("Gradient at point [2.0, 0.5]:", grad_val)
print("Analytical: [4.0, cos(0.5)] = [4.0, 0.87758]")

Gradient at point [2.0, 0.5]: [4.         0.87758256]
Analytical: [4.0, cos(0.5)] = [4.0, 0.87758]


In [9]:
# %% [markdown]
# #7. Performance benchmark: vectorised vs. loop

# %%

import time

large_arr = np.random.randn(10_000_000)

# Vectorised mean
start = time.time()
mean_vec = np.mean(large_arr)
time_vec = time.time() - start

# Pure Python loop
start = time.time()
s = 0.0
for v in large_arr:
    s += v
mean_loop = s / len(large_arr)
time_loop = time.time() - start

print(f"Vectorised mean: {mean_vec:.6f} (time: {time_vec:.4f} s)")
print(f"Loop mean      : {mean_loop:.6f} (time: {time_loop:.4f} s)")
print(f"Speedup: {time_loop / time_vec:.1f}x")


Vectorised mean: 0.000107 (time: 0.0094 s)
Loop mean      : 0.000107 (time: 1.4475 s)
Speedup: 154.4x


In [10]:
# Define pipelines separately
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy='mean')),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy='constant', fill_value=-1)),
    ("encoder", OneHotEncoder(handle_unknown='ignore'))
])

# Wrap them using lambda-style transformers (simple approach)
class NumSelector:
    def transform(self, X):
        return X[:, num_idx]
    def fit(self, X, y=None):
        return self

class CatSelector:
    def transform(self, X):
        return X[:, cat_idx]
    def fit(self, X, y=None):
        return self

# Combine using FeatureUnion
union = FeatureUnion([
    ("num", Pipeline([
        ("selector", NumSelector()),
        ("num_pipe", num_pipeline)
    ])),
    ("cat", Pipeline([
        ("selector", CatSelector()),
        ("cat_pipe", cat_pipeline)
    ]))
])

X_union = union.fit_transform(data)

print("FeatureUnion output shape:", X_union.shape)

FeatureUnion output shape: (2392, 34)


In [12]:
# Simulated classification example
y_true = np.array([1, 0, 1, 1, 0, 1, 0])
y_pred = np.array([1, 0, 0, 1, 0, 1, 1])

print("Accuracy :", accuracy(y_true, y_pred))
print("Precision:", precision(y_true, y_pred))
print("Recall   :", recall(y_true, y_pred))
print("F1 Score :", f1(y_true, y_pred))

print("\nConfusion Matrix:\n", confusion_matrix(y_true, y_pred))

y_true_reg = np.array([3.0, 2.5, 4.0])
y_pred_reg = np.array([2.8, 2.7, 3.9])

print("MSE:", mse(y_true_reg, y_pred_reg))

# Simulated probability scores
y_true = np.array([0, 0, 1, 1])
y_scores = np.array([0.1, 0.4, 0.35, 0.8])

fpr, tpr, thresholds = roc_curve(y_true, y_scores)
roc_auc = auc(fpr, tpr)

print("FPR:", fpr)
print("TPR:", tpr)
print("AUC:", roc_auc)

Accuracy : 0.7142857142857143
Precision: 0.75
Recall   : 0.75
F1 Score : 0.75

Confusion Matrix:
 [[2 1]
 [1 3]]
MSE: 0.030000000000000054
FPR: [0.  0.  0.5 0.5 1. ]
TPR: [0.  0.5 0.5 1.  1. ]
AUC: 0.75


## Summary

This notebook demonstrated:
- Loading CSV with `io.load_csv`
- Preprocessing (imputation, scaling, one‑hot encoding) using `preprocessing`
- Statistics (`mean`, `std`, `histogram`, `quantile`) from `stats`
- Sorting, binary search, ranking, percentile using `sort_search` and `rank`
- Finite‑difference gradient estimation with `optim`
- Benchmarking vectorised NumPy vs. pure Python loops